# KTTS single-speaker adaptation of Bolbosh Matcha-TTS
Fine-tunes the MIT-licensed GAASH Lab Kashmiri checkpoint on the CC BY-NC 3.0 KTTS corpus. Outputs are non-commercial and require attribution.

In [ ]:
!nvidia-smi
!git clone -q --depth 1 https://github.com/sameerahmad86232/cipher.git /kaggle/working/cipher
!wget -q https://github.com/gaash-lab/Bolbosh/archive/refs/heads/main.zip -O /kaggle/working/bolbosh.zip
!unzip -q /kaggle/working/bolbosh.zip -d /kaggle/working
%cd /kaggle/working/Bolbosh-main
!pip install -q 'numpy<2' Cython soundfile scipy pyarrow huggingface_hub lightning hydra-core hydra-colorlog rootutils rich torchmetrics tensorboard librosa einops inflect Unidecode conformer diffusers peft jiwer
!pip install -q git+https://github.com/abdulmuizz0903/KashmiriNormalizer
!python setup.py build_ext --inplace -q

In [ ]:
!python /kaggle/working/cipher/scripts/prepare-ktts-matcha.py --repo-dir /kaggle/working/Bolbosh-main --output /kaggle/working/ktts-matcha --local-root /kaggle/input
from pathlib import Path
import json
report_path = Path('/kaggle/working/ktts-matcha/preparation-report.json')
report = json.loads(report_path.read_text()) if report_path.exists() else {'accepted':0,'hours':0,'error':'preparation report missing'}
print(report)
ready_for_training = report['accepted'] >= 2500 and report['hours'] >= 3.5
print('Dataset ready for training:', ready_for_training)

In [ ]:
from huggingface_hub import hf_hub_download
import torch
try:
    base = hf_hub_download('GAASH-Lab/Matcha-TTS-Kashmiri', 'model.ckpt')
    checkpoint = torch.load(base, map_location='cpu', weights_only=False)
    checkpoint['epoch'] = -1
    checkpoint['global_step'] = 0
    checkpoint['optimizer_states'] = []
    checkpoint['lr_schedulers'] = []
    checkpoint.pop('loops', None)
    warmstart = '/kaggle/working/bolbosh-ktts-warmstart.ckpt'
    torch.save(checkpoint, warmstart)
    print('Prepared weights-only warm start:', warmstart)
except Exception as error:
    Path('/kaggle/working/warmstart-error.txt').write_text(repr(error))
    print('Warm-start preparation failed:', repr(error))

In [ ]:
import subprocess, sys
from pathlib import Path
log_path = Path('/kaggle/working/tts-training.log')
command = [sys.executable, 'matcha/train.py', 'data=kashmiri', 'model=matcha', 'trainer=gpu', 'logger=tensorboard', 'ckpt_path=/kaggle/working/bolbosh-ktts-warmstart.ckpt', 'train=true', 'test=false', 'run_name=ktts_voice_v1', 'data.batch_size=16', 'data.num_workers=2', 'trainer.max_epochs=30', 'trainer.precision=16-mixed', '+trainer.accumulate_grad_batches=2', 'paths.output_dir=/kaggle/working/bolbosh-ktts-output']
with log_path.open('w') as log:
    result = subprocess.run(command, cwd='/kaggle/working/Bolbosh-main', stdout=log, stderr=subprocess.STDOUT, text=True)
print('Training exit code:', result.returncode)
print(log_path.read_text(errors='replace')[-12000:])

In [ ]:
from pathlib import Path
import shutil
root = Path('/kaggle/working/bolbosh-ktts-output')
checkpoints = sorted(root.rglob('*.ckpt'), key=lambda p: p.stat().st_mtime)
print('Checkpoints found:', len(checkpoints))
release = Path('/kaggle/working/kashmiri-bolbosh-ktts-v1')
release.mkdir(exist_ok=True)
if checkpoints: shutil.copy2(checkpoints[-1], release / 'model.ckpt')
report_path = Path('/kaggle/working/ktts-matcha/preparation-report.json')
if report_path.exists(): shutil.copy2(report_path, release / 'preparation-report.json')
(release / 'LICENSE-DATA.txt').write_text('KTTS data: CC BY-NC 3.0. Non-commercial use only. DOI: 10.17632/5c4dcvxdmb.2\n')
shutil.make_archive('/kaggle/working/kashmiri-bolbosh-ktts-v1', 'zip', release)
print('Artifact:', '/kaggle/working/kashmiri-bolbosh-ktts-v1.zip')